# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution observations

The analysis uses aggregated content-level performance measurements and query-derived signals. The distributions are heavy-tailed: some content items have much higher historical impressions and kept impressions than others.

The dataset includes impression counts, clicks, position measurements, visible-query counts, rare-query share, anonymous-query share, and kept impressions. These signals describe observed performance and query coverage; they do not establish why an item performed well or poorly.

The analysis focuses on aggregate patterns and uses the results for decision-support, not causal claims.


In [4]:
import pandas as pd
import numpy as np

In [5]:
# ML-06: Distribution summary from the full-release analysis.
distribution_summary = pd.DataFrame([
    {"field": "imp_prev30", "count": 136252, "median": 829, "maximum": 1369821},
    {"field": "imp_last30", "count": 136252, "median": 249, "maximum": 615012},
    {"field": "clk_last30", "count": 136252, "median": 1, "maximum": 3817},
    {"field": "pos_last30", "count": 132139, "median": 14.29, "maximum": 579},
    {"field": "visible_queries", "count": 117254, "median": 7, "maximum": 7889},
    {"field": "kept_impressions", "count": 117254, "median": 226, "maximum": 606884},
])

display(distribution_summary)

,field,count,median,maximum
0,imp_prev30,136252,829.00,1369821
1,imp_last30,136252,249.00,615012
2,clk_last30,136252,1.00,3817
3,pos_last30,132139,14.29,579
4,visible_queries,117254,7.00,7889
5,kept_impressions,117254,226.00,606884


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1 — Previous-window visibility**

**Verdict: MIXED**

Items with at least 500 previous-window impressions had a decline rate of 90.12%, compared with 84.72% for items below 500 impressions. The higher-visibility group had a higher observed decline rate by approximately 5.40 percentage points.

Both groups had high decline rates, so previous-window visibility alone does not distinguish reliably between declining and non-declining content. This is a descriptive result, not evidence of causation.


**Signal 2 — Latest impressions**

**Verdict: CONFIRMED (by definition)**

Items with zero latest-window impressions had a decline rate of 100% (4,113 items). Items with nonzero latest-window impressions had a decline rate of 87.68% (132,139 items).

This pattern is partly expected because latest-window impressions are used to calculate the decline label. Therefore, this is not independent evidence that zero impressions predict future declines. Zero-impression items should be investigated, but the result should not be interpreted as a future prediction.


**Signal 3 — Position groups**

**Verdict: MIXED**

The observed decline rate increased across worse position groups: 71.69% for positions 1–3, 80.52% for positions 4–10, 89.68% for positions 11–20, and 93.78% for positions above 20.

This pattern suggests an association between worse positions and higher observed decline rates in this dataset. However, position data is missing for some items, and the analysis does not prove that worse position causes declining impressions.


In [8]:
signal1 = pd.DataFrame([
    {"visibility_group": "500_plus", "items": 84180, "decline_rate": 0.901212},
    {"visibility_group": "under_500", "items": 52072, "decline_rate": 0.847192},
])

display(signal1)
print(
    "Difference in decline rates:",
    round((0.901212 - 0.847192) * 100, 2),
    "percentage points"
)

signal2 = pd.DataFrame([
    {"latest_group": "nonzero_latest_impressions", "items": 132139, "decline_rate": 0.876849},
    {"latest_group": "zero_latest_impressions", "items": 4113, "decline_rate": 1.000000},
])

display(signal2)

signal3 = pd.DataFrame([
    {"position_group": "1–3", "items": 2543, "decline_rate": 0.716870},
    {"position_group": "4–10", "items": 46633, "decline_rate": 0.805202},
    {"position_group": "11–20", "items": 31840, "decline_rate": 0.896765},
    {"position_group": "Over 20", "items": 51123, "decline_rate": 0.937758},
])

display(signal3)

,visibility_group,items,decline_rate
0,500_plus,84180,0.901212
1,under_500,52072,0.847192


Difference in decline rates: 5.4 percentage points


,latest_group,items,decline_rate
0,nonzero_latest_impressions,132139,0.876849
1,zero_latest_impressions,4113,1.000000


,position_group,items,decline_rate
0,1–3,2543,0.716870
1,4–10,46633,0.805202
2,11–20,31840,0.896765
3,Over 20,51123,0.937758


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag-linked test — Zero latest-window impressions**

**Assumption being tested:** Content with zero latest-window impressions should be investigated because it may have lost visibility or stopped receiving traffic.

**Evidence:** There were 4,113 items with zero latest-window impressions, and their observed decline rate was 100%. Their median previous-window impressions were 180.

**Verdict: MIXED**

Zero latest-window impressions are a useful investigation signal, but the 100% decline rate is expected from the way the target label is defined. The result does not establish why impressions became zero or whether a particular FlyRank flag correctly identifies a problem. Tracking gaps, missing measurements, or other changes should be checked before taking action.


In [9]:
import pandas as pd

# ML-06: Flag-linked test — zero latest-window impressions
flag_test = pd.DataFrame([
    {
        "signal": "Zero latest-window impressions",
        "items": 4113,
        "decline_rate": 1.000000,
        "median_previous_impressions": 180,
    }
])

display(flag_test)

print("Verdict: MIXED")
print(
    "Zero latest-window impressions are a useful investigation signal, "
    "but latest-window impressions are also used to define the decline label."
)
print(
    "This result does not establish why impressions became zero "
    "or prove that a particular flag correctly identifies a problem."
)


,signal,items,decline_rate,median_previous_impressions
0,Zero latest-window impressions,4113,1.0,180


Verdict: MIXED
Zero latest-window impressions are a useful investigation signal, but latest-window impressions are also used to define the decline label.
This result does not establish why impressions became zero or prove that a particular flag correctly identifies a problem.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit suggests that content review should consider multiple signals rather than relying on a single threshold.

* **Previous-window visibility:** Higher-visibility items showed a higher observed decline rate, but both groups had high decline rates.
* **Latest impressions:** Zero latest-window impressions are a useful reason to investigate an item, but this signal overlaps with the target definition.
* **Position:** Worse position groups showed higher observed decline rates in this dataset, but this association does not prove causation.

Missing measurements, tracking changes, seasonality, and differences in measurement windows may affect the findings. These results support investigation and review prioritization; they do not prove that refreshing content will improve performance.


In [10]:
print("Practical implications")
print("- Use multiple signals to prioritize content for human review.")
print("- Previous-window visibility alone is not enough to assess decline.")
print("- Zero latest-window impressions should trigger investigation,")
print("  but this signal overlaps with the decline-label definition.")
print("- Worse position groups showed higher observed decline rates.")
print("- Missing data, tracking changes, and seasonality may affect results.")
print("- These findings are descriptive and do not prove causation.")


Practical implications
- Use multiple signals to prioritize content for human review.
- Previous-window visibility alone is not enough to assess decline.
- Zero latest-window impressions should trigger investigation,
  but this signal overlaps with the decline-label definition.
- Worse position groups showed higher observed decline rates.
- Missing data, tracking changes, and seasonality may affect results.
- These findings are descriptive and do not prove causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.